# Asteria — Mixed100K V2: Training and Evaluation

English Colab workflow for TikTok TechJam 2026, Problem 5. [Repository and README](https://github.com/sunzk111/ai-image-detector) · [Model release](https://github.com/sunzk111/ai-image-detector/releases/tag/model).

**Default: evaluate the released model.** Leave `TRAIN_FROM_SCRATCH = False`; training-data preparation and training will be skipped. To train a new model, set it to `True` in a fresh checkout. The model architecture, training hyperparameters, and augmentation policy are unchanged.

Read each section before running. Evaluation-data archives require roughly 28 GB to download, plus extraction space; the model requires about 1 GB. If your evaluation images already exist, set `DOWNLOAD_EVAL_DATA = False`. No datasets or model weights are embedded in this notebook.

Use a GPU for training/full evaluation. Data downloads need no GPU, but changing the Colab runtime may remove `/content`: back up needed files first. Outputs and private runtime metadata have been cleared.

## 1. Choose the workflow and open the project

The setup reuses an existing project without overwriting it. A fresh checkout obtains the current `main` branch, which contains this English notebook. Do not add access tokens to shared cells.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

TRAIN_FROM_SCRATCH = False
DOWNLOAD_EVAL_DATA = True  # Set False when the benchmark images are already present.
PROJECT = Path('/content/ai_image_detector')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/sunzk111/ai-image-detector.git', str(PROJECT)], check=True)
assert (PROJECT / 'evaluate.py').is_file(), 'Project missing or incomplete; choose the correct PROJECT directory.'
os.chdir(PROJECT)
EVAL_CONFIG = 'config.mixed100k.yaml' if TRAIN_FROM_SCRATCH else 'config.release.yaml'
CHECKPOINT = PROJECT / 'outputs/dinov2_mixed100k_v2/best.pt'
print('Project:', PROJECT)
print('Workflow:', 'train a new model' if TRAIN_FROM_SCRATCH else 'evaluate the released model')


## 2. Install dependencies

Colab does not require `.venv` activation. The original run used A100 40 GB, PyTorch 2.11.0+cu128, and BF16. The dependency file specifies minimum versions, not a frozen environment. First model initialization also downloads DINOv2 from Hugging Face.


In [ ]:
%pip install -r requirements.txt
%pip install kagglehub modelscope-hub


## 3. Download the released weights — evaluation workflow only

Downloads [best.pt](https://github.com/sunzk111/ai-image-detector/releases/download/model/best.pt) and its [configuration](https://github.com/sunzk111/ai-image-detector/releases/download/model/config.mixed100k.yaml) from the `model` release. The configuration is saved as **`config.release.yaml`** to preserve the repository's original training configuration.

Existing files are checked against the release SHA-256 values. A different local checkpoint is never replaced: preserve it and use a fresh project directory instead. Interrupted downloads use `.part` files and can resume. If a checksum fails, do not load that file.

The released checkpoint's decision threshold is **0.000005**. New training runs calibrate their own threshold and should not inherit this manual value. This cell is skipped when `TRAIN_FROM_SCRATCH = True`.


In [ ]:
import hashlib

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not TRAIN_FROM_SCRATCH:
    release_url = 'https://github.com/sunzk111/ai-image-detector/releases/download/model/'
    assets = [
        ('best.pt', CHECKPOINT, 'f1e2d5470db116f59a64f65d8e3b7bccf2fd5fdec7c163023dbbd7b6d35b9508'),
        ('config.mixed100k.yaml', PROJECT / EVAL_CONFIG, 'feb1765f6d18db1f0853f22669c7fdb8fd79204380e43774eb308138557849a4'),
    ]
    for asset_name, destination, expected_hash in assets:
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            assert sha256_file(destination) == expected_hash, f'Different existing file: {destination}. Preserve it; use a fresh project directory.'
        else:
            partial = destination.with_name(destination.name + '.part')
            subprocess.run(['curl', '-fL', '--retry', '3', '-C', '-', release_url + asset_name, '-o', str(partial)], check=True)
            assert sha256_file(partial) == expected_hash, f'Checksum mismatch: {partial}; do not load it.'
            partial.rename(destination)
        print('Verified:', destination)
else:
    print('Skipped release downloads: training from scratch.')


## 4. Download the evaluation-only images

The benchmark is COCO val2017 (**4,998 real**) plus DALL·E Advanced/DALLE3 (**8,843 AI**). **Do not train on these images.** The full upstream archives are downloaded before subset extraction, which can take considerable time and disk space. Downloads are resumable through ModelScope; `unzip -n` does not replace existing extracted files.

If you already have the data, disable `DOWNLOAD_EVAL_DATA`. The default root is `data/wildfake_eval`, containing immediate `coco/` and `DALLE/` class folders, with nested image folders supported. If using a different location, edit `data.validation_demo.path` in the active evaluation configuration before the count check. Keep the default path when preparing training data, or supply the same directory via `--test-root`.


In [ ]:
if DOWNLOAD_EVAL_DATA:
    (PROJECT / 'data/wildfake_raw').mkdir(parents=True, exist_ok=True)
    (PROJECT / 'data/wildfake_eval').mkdir(parents=True, exist_ok=True)
    download_env = dict(os.environ, MODELSCOPE_DOWNLOAD_PARALLEL_WORKERS='4')
    for archive, labels in [
        ('Images/Diffusion_based/DALLE.zip', 'label_csv_files/dalle3.csv'),
        ('Images/Real/coco.zip', 'label_csv_files/real_coco.csv'),
    ]:
        subprocess.run(['ms-hub', 'download', 'hy2628982280/WildFake', archive, labels, '--repo-type', 'dataset', '--local-dir', 'data/wildfake_raw', '--max-workers', '2'], env=download_env, check=True)
    subprocess.run(['unzip', '-n', '-q', 'data/wildfake_raw/Images/Diffusion_based/DALLE.zip', 'DALLE/Advanced/DALLE3/*', '-d', 'data/wildfake_eval'], check=True)
    subprocess.run(['unzip', '-n', '-q', 'data/wildfake_raw/Images/Real/coco.zip', '*val2017/*', '-d', 'data/wildfake_eval'], check=True)
else:
    print('Download skipped: using existing evaluation images.')


In [ ]:
from collections import Counter
import yaml
from dataset import samples_from_imagefolder

with open(EVAL_CONFIG, encoding='utf-8') as handle:
    evaluation_config = yaml.safe_load(handle)
benchmark_samples = samples_from_imagefolder(evaluation_config['data']['validation_demo'], evaluation_config['data']['class_to_label'])
counts = Counter(sample.label for sample in benchmark_samples)
print('Real:', counts[0], '| AI:', counts[1], '| Total:', len(benchmark_samples))
assert counts == {0: 4998, 1: 8843}, 'Benchmark count mismatch: inspect class folders and extraction before continuing.'


## 5. Prepare Mixed100K — training workflow only

Skipped by default. Creates **100,000 training + 4,000 internal-validation images**: CIFAKE and SID_Set contribute 10,000 training images each; WildFake contributes 80,000. Training is balanced between real and AI. WildFake COCO/DALL·E archives and SID_Set label 2 are excluded.

Use the repository's `config.mixed100k.yaml` template to preserve the original training and calibration settings. Re-running resumes from the preparation cache; do not change seed/quotas in an existing cache. Complete caches can use `--existing-only`. The demonstration images should already exist so the builder can reject decoded-RGB exact overlap; this is not near-duplicate detection.

Preparation regenerates `config.mixed100k.yaml` and the optional smoke configuration. It resets `output_dir` and `training.resume`; make any new-run or resume edits after this step. [Detailed source quotas and storage notes](https://github.com/sunzk111/ai-image-detector/blob/main/MIXED100K_README.md).


In [ ]:
if TRAIN_FROM_SCRATCH:
    subprocess.run([sys.executable, '-u', 'prepare_mixed_dataset.py', '--base-config', 'config.mixed100k.yaml', '--workers', '8', '--test-root', evaluation_config['data']['validation_demo']['path']], check=True)
else:
    print('Skipped: the released model does not require training data.')


## 6. Train a new model — training workflow only

Original settings: 10 epochs, batch size 16, accumulation 1, backbone/head learning rates 1e-5/1e-4, AdamW, 10% warmup and cosine decay. The first epoch freezes the backbone. Clean/degraded paired training uses BCE for each view plus feature consistency; the degradation curriculum includes strong blur and JPEG q30. No extra blur-specific oversampling is added here.

The 4,000 internal-validation images are split by fixed IDs into 1,000 calibration and 3,000 model-selection images. Each checkpoint stores its corresponding threshold.

The guard below stops before overwriting an existing `best.pt` or `last.pt`. If needed, change `output_dir` in `config.mixed100k.yaml` to a new experiment folder after preparation, then run this cell. Reduce batch size if GPU memory is insufficient. This notebook runs fresh training; for resume instructions, see the README.


In [ ]:
if TRAIN_FROM_SCRATCH:
    with open('config.mixed100k.yaml', encoding='utf-8') as handle:
        training_config = yaml.safe_load(handle)
    training_output = Path(training_config['output_dir'])
    assert not any((training_output / name).exists() for name in ('best.pt', 'last.pt')), 'Existing weights: choose a new output_dir before training.'
    assert not training_config['training'].get('resume'), 'This notebook section is for fresh training; see README for resume.'
    CHECKPOINT = (training_output / 'best.pt').resolve()
    subprocess.run([sys.executable, 'train.py', '--config', 'config.mixed100k.yaml'], check=True)
else:
    print('Skipped training: evaluating the released checkpoint.')


## 7. Evaluate the model

Both workflows use this cell. The released model uses `config.release.yaml`; a newly trained model uses `config.mixed100k.yaml` and its own checkpoint threshold. Check the printed `Decision threshold` line: the release operating point is **0.000005**, but a new model can have a different calibrated threshold. `--threshold` would affect only the report, not update the checkpoint.

Set `SMOKE_EVALUATION = True` for a 64-image check of clean, JPEG q30, and blur radius 2. Leave it `False` for all 13,841 images under all 16 conditions. A smoke result is not a complete benchmark.

Each run creates a new `evaluation_runs/<run_id>/` beside the checkpoint. Full evaluation also refreshes top-level `robustness.csv/json`. The diagnostic sweep does not automatically select a threshold.

**Reporting caveat:** the released 5e-6 threshold was manually adjusted using the demonstration benchmark. These images were not used for gradient training, but the reported results are development-benchmark results, not an untouched final test. The original `evaluate.py` remains unchanged; the optional `predict.py` section below handles unlabeled image-directory → JSON inference.


In [ ]:
SMOKE_EVALUATION = False
assert CHECKPOINT.is_file(), f'Missing checkpoint: {CHECKPOINT}'
command = [sys.executable, 'evaluate.py', '--config', EVAL_CONFIG, '--checkpoint', str(CHECKPOINT), '--source', 'validation_demo']
if SMOKE_EVALUATION:
    command += ['--conditions', 'clean', 'jpeg_q30', 'blur_sigma2', '--max-samples', '64']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


## 8. Inspect the latest reports

`robustness.csv/json` stores per-condition metrics; `predictions.csv` stores per-image logits, scores, labels, and errors; `threshold_sweep.csv` and `score_distributions.csv` provide diagnostics; `run_status.json` records completion/failure. Accuracy is threshold-dependent; AUROC measures ranking.

The reported release achieved **96.55% clean accuracy**, **95.27% mean accuracy over 15 transformed conditions**, and **91.03% accuracy under the strongest blur**. Exact scores may vary with hardware, precision, library versions, and dataset contents.

To compare thresholds without running the model again, use `python evaluate.py --from-predictions PATH/TO/predictions.csv --threshold 0.000005`. See the README for the complete command. Do not repeatedly tune on a final test set.


In [ ]:
import csv
import json

runs = sorted((CHECKPOINT.parent / 'evaluation_runs').glob('*/run_status.json'))
assert runs, 'No evaluation run found.'
latest_run = runs[-1].parent
status = json.loads(runs[-1].read_text())
print('Latest run:', latest_run)
print('Status:', status.get('status'))
assert status.get('status') == 'complete', 'Latest run is incomplete; inspect run_status.json.'
with (latest_run / 'robustness.csv').open(newline='') as handle:
    for row in csv.DictReader(handle):
        print(f"{row['condition']:25} accuracy={float(row['accuracy']):.4f} AUROC={float(row['auroc']):.4f} F1={float(row['f1']):.4f}")


## 9. Optional: predict an unlabeled image directory as JSON

Upload the companion **`predict.py`** into the project root if it is not already in your checkout. This standalone script does not modify or call `evaluate.py`. For JSON inference only, run sections 1–3, skip sections 4–8, and then use this section. No evaluation dataset, training data, labels, or class-folder structure is required.

Set `RUN_JSON_INFERENCE = True`, set `INPUT_IMAGES` to your image directory, and choose a new `JSON_OUTPUT` filename. Subdirectories are scanned recursively; supported formats are JPG/JPEG, PNG, WEBP, BMP, and TIF/TIFF.

Output is an array of objects with **`image_path`** (relative to the input directory, using forward slashes) and **`pred`** (continuous AI score from 0 to 1). It does not threshold the scores. The released 5e-6 operating point can be used separately for binary decisions; never replace the JSON score with a 0/1 label. Existing output files are not overwritten, and unreadable images cause an explicit error.


In [ ]:
RUN_JSON_INFERENCE = False
INPUT_IMAGES = Path('/content/input_images')
JSON_OUTPUT = PROJECT / 'predictions.json'
if RUN_JSON_INFERENCE:
    assert (PROJECT / 'predict.py').is_file(), 'Upload the companion predict.py into the project root first.'
    subprocess.run([sys.executable, 'predict.py', '--config', EVAL_CONFIG, '--checkpoint', str(CHECKPOINT), '--input-dir', str(INPUT_IMAGES), '--output-json', str(JSON_OUTPUT)], check=True)
    import json
    predictions = json.loads(JSON_OUTPUT.read_text(encoding='utf-8'))
    print('Images:', len(predictions))
    print(predictions[:3])
else:
    print('Optional JSON inference disabled. Set paths and RUN_JSON_INFERENCE to enable it.')


## 10. Optional: back up code, weights, and reports to Drive

Disabled by default. Set `BACKUP_TO_DRIVE = True` to mount your own Drive and create a timestamped backup. It **excludes `data/` and `.git/`**, but includes model weights and reports. Back up datasets separately if you want to avoid downloading them again. This archive is for your storage, not a GitHub upload.

Confirm the backup completed before disconnecting/deleting the runtime. `/content` is temporary. To restore later, extract your trusted archive into a fresh runtime, not over an existing project.


In [ ]:
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    from datetime import datetime, timezone
    drive.mount('/content/drive')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    backup = Path('/content/drive/MyDrive') / f'ai_image_detector_backup_{stamp}.tar.gz'
    subprocess.run(['tar', f'--exclude={PROJECT.name}/data', f'--exclude={PROJECT.name}/.git', '--exclude=__pycache__', '-czf', str(backup), '-C', str(PROJECT.parent), PROJECT.name], check=True)
    assert backup.is_file() and backup.stat().st_size > 0
    print('Backup completed:', backup, '| bytes:', backup.stat().st_size)
else:
    print('Drive backup disabled. Save needed weights and data before ending the runtime.')
